# Burn Model Analysis for `ci_payment_details_2.csv`

This notebook analyzes what a good burn model should look like for the revised payment-detail data. It focuses on the `_months` fields and treats each row as one payment event for one contract item.

## Core Conclusion

The best model depends on the job. For allocating a known item total across the item timeline, the interval-weighted model is structurally best because it conserves item totals. For predicting the typical row, a robust multiplicative model on top of `CI_BURN_RATE_MONTHS` performs better on median error because payment events are bursty and payment dates do not behave like pure accrual boundaries. A practical model should therefore be hierarchical and probabilistic: use item-level scale, model row-level share/ratio with sequence and timing features, and return prediction intervals rather than a single deterministic burn.

## Dataset and Modeling Setup

| Metric | Value |
| --- | --- |
| Rows | 16,435 |
| Items | 875 |
| Train items / rows | 679 / 12,784 |
| Test items / rows | 196 / 3,651 |
| Median THIS_BURN | 65,215.75 |
| Median monthly baseline | 207,983.37 |
| Median actual/monthly-rate ratio | 0.2240 |
| Mean actual/monthly-rate ratio | 0.5650 |
| Rows within +/-25% of monthly baseline | 1,574 (9.58%) |
| Rows below -100% delta | 126 |
| Rows above +100% delta | 986 |
| Negative burn rows | 126 |

Rows are split by `ITEMID` into train/test groups using a deterministic hash. This prevents rows from the same item appearing in both train and test. That matters because item-level scale is repeated across every row for an item.

In [ ]:
import csv
from pathlib import Path

CSV_PATH = Path('ci_payment_details_2.csv')
with CSV_PATH.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)

print(f'Rows: {len(rows):,}')
print(f'Fields: {reader.fieldnames}')

## Important Leakage / Scope Caveat

`CI_BURN_RATE_MONTHS` appears to be derived from the item total burn divided by modeled months. That makes it an excellent normalization field for retrospective analysis, but it can leak future information if the task is to forecast an item before its total burn is known.

So there are two different modeling problems:

1. **Retrospective allocation / anomaly detection:** Given item scale and duration, estimate whether each payment event is unusually high or low. This file supports that well.
2. **Forward forecasting before item completion:** Estimate future burn without knowing final item total. This file alone is not enough; it needs item quantity, unit price, item type, contract, project, and schedule covariates.

## Why the Current Monthly Delta Is Not a Full Model

`BURN_DELTA_MONTHS_PERCENT` compares each payment event to a full monthly burn rate. But each row is a payment event, not a guaranteed one-month period. The median row is only about 22% of the monthly baseline, so most rows look negative against a full-month comparator. That does not automatically mean underperformance; it often means the row represents a smaller accounting/payment slice.

## Model Comparison on Held-Out Items

| Model | Bias | MAE | RMSE | Median AE | Mean abs err / monthly rate | Median abs err / monthly rate | MAPE nonzero actuals |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Current monthly-rate per row | -267,097.16 | 489,832.01 | 1,166,404.41 | 177,614.95 | 0.7873 | 0.8199 | 71.1818 |
| Global median row/month ratio | 269,364.21 | 349,904.32 | 1,650,590.74 | 48,444.70 | 0.4596 | 0.1628 | 15.8844 |
| Global mean row/month ratio | 33,405.33 | 383,526.00 | 1,316,764.49 | 103,300.82 | 0.5488 | 0.4334 | 40.0952 |
| Sequence-bucket median ratio | 268,022.88 | 344,020.42 | 1,630,093.71 | 49,394.62 | 0.4568 | 0.1618 | 9.6137 |
| Calendar-progress median ratio | 269,325.46 | 348,250.14 | 1,643,937.12 | 47,846.77 | 0.4555 | 0.1646 | 12.1713 |
| Sequence x item-size median ratio | 263,200.39 | 359,030.13 | 1,680,291.39 | 52,995.00 | 0.4499 | 0.1754 | 8.8391 |
| Interval-weighted linear allocation | -0.59 | 304,312.59 | 1,414,225.75 | 62,789.06 | 0.6123 | 0.1997 | 380.6772 |
| Interval allocation x seq residual median | -417,109.72 | 541,351.37 | 2,323,935.40 | 64,274.41 | 0.8456 | 0.2646 | 199.9611 |

## Interpreting the Model Comparison

- The current monthly-rate-per-row baseline is systematically too high for most individual rows. It has a large negative bias because predicted burn is one full month of burn for every payment event.
- The global median row/month ratio has low median absolute error because it predicts a typical payment event as a fraction of monthly burn, not a full month.
- The global mean ratio is less biased in dollars, but worse for the median row because the positive tail is large.
- Sequence and calendar-progress median-ratio models make only modest row-level improvements. That means the dominant signal is item scale plus heavy-tailed payment-event noise, not a smooth deterministic curve.
- The interval-weighted linear allocation is the best structural allocation model: it nearly reconciles item totals, but row-level errors remain large because payment dates are not clean work-accrual period boundaries.

## Sequence Shape

This table describes actual burn as a ratio of the monthly baseline by payment-event position. A value of `0.25` means the median payment in that bucket is one quarter of the item monthly burn rate.

| Sequence fraction bucket | Rows | P25 actual/monthly | Median actual/monthly | P75 actual/monthly | Median delta % | Negative burn share |
| --- | --- | --- | --- | --- | --- | --- |
| <= 5% | 1,234 | 0.0934 | 0.2523 | 0.9116 | -74.77 | 0.08% |
| 5%-10% | 758 | 0.0962 | 0.2296 | 0.6314 | -77.04 | 0.26% |
| 10%-20% | 1,657 | 0.1130 | 0.2362 | 0.7260 | -76.38 | 0.36% |
| 20%-30% | 1,508 | 0.1148 | 0.2566 | 0.6475 | -74.34 | 0.46% |
| 30%-40% | 1,586 | 0.1284 | 0.2579 | 0.7068 | -74.21 | 0.06% |
| 40%-50% | 1,717 | 0.1218 | 0.2598 | 0.7875 | -74.02 | 0.41% |
| 50%-60% | 1,391 | 0.1177 | 0.2389 | 0.6103 | -76.11 | 0.65% |
| 60%-70% | 1,524 | 0.1287 | 0.2476 | 0.6373 | -75.24 | 0.46% |
| 70%-80% | 1,570 | 0.1139 | 0.2174 | 0.5383 | -78.26 | 0.51% |
| 80%-90% | 1,595 | 0.1020 | 0.2053 | 0.5220 | -79.47 | 1.13% |
| 90%-95% | 687 | 0.0683 | 0.1486 | 0.3468 | -85.14 | 1.60% |
| > 95% | 1,208 | 0.0319 | 0.1132 | 0.3384 | -88.68 | 4.06% |

## Calendar-Interval Allocation

A natural linear model is: `predicted burn = CI_BURN_RATE_MONTHS * interval_days / 30`, where the first event receives 30 days and later events receive days since prior payment. This model aligns with the `(DAYS_BETWEEN + 30) / 30` construction.

| Payment position | Rows | Median interval days | P25 actual/interval pred | Median actual/interval pred | P75 actual/interval pred | P95 actual/interval pred |
| --- | --- | --- | --- | --- | --- | --- |
| First payment | 875 | 30.00 | 0.0997 | 0.2824 | 1.1865 | 5.0400 |
| Middle payments | 14,685 | 2.00 | 0.8683 | 2.4005 | 5.5824 | 23.1305 |
| Last payment | 875 | 21.00 | 0.0401 | 0.3759 | 1.7462 | 8.7251 |

## Item-Total Reconciliation of Interval Model

| Metric | Value |
| --- | --- |
| Median absolute item total error | 0.099549 |
| Max absolute item total error | 25,731.573884 |
| Items within $1 of item total | 637 |
| Items within $100 of item total | 700 |

This is why the interval-weighted model is useful even when it is not the best row-level point predictor. It allocates the known item total across time in a way that is internally coherent.

## Extreme Positive Rows

| ITEMID | Seq | Num | Date | Interval days | Monthly base | This burn | Delta % |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 19672 | 2 | 15 | 2017-05-06 06:00:00.000 | 44.00 | 107,615.57 | 2,081,965.25 | 1,834.63 |
| 19670 | 2 | 12 | 2017-05-06 06:00:00.000 | 44.00 | 100,318.47 | 1,922,666.51 | 1,816.56 |
| 61751 | 1 | 11 | 2020-05-15 06:00:00.000 | 30.00 | 64,916.35 | 1,229,991.00 | 1,794.73 |
| 44983 | 3 | 11 | 2019-11-16 07:00:00.000 | 91.04 | 39,701.77 | 692,580.00 | 1,644.46 |
| 57300 | 5 | 9 | 2020-07-27 06:00:00.000 | 32.00 | 196,851.90 | 3,381,031.00 | 1,617.55 |
| 68867 | 1 | 8 | 2021-08-21 06:00:00.000 | 30.00 | 26,145.42 | 424,836.73 | 1,524.90 |
| 75397 | 2 | 13 | 2022-03-19 06:00:00.000 | 62.96 | 39,326.76 | 630,099.70 | 1,502.22 |
| 80920 | 1 | 7 | 2022-04-16 06:00:00.000 | 30.00 | 123,351.81 | 1,974,747.34 | 1,500.91 |
| 71522 | 1 | 14 | 2021-05-21 06:00:00.000 | 30.00 | 146,645.06 | 2,179,270.00 | 1,386.08 |
| 36218 | 32 | 68 | 2019-01-18 07:00:00.000 | 3.00 | 11,463.70 | 166,792.03 | 1,354.96 |
| 73938 | 1 | 13 | 2021-08-21 06:00:00.000 | 30.00 | 47,238.51 | 681,816.51 | 1,343.35 |
| 57300 | 6 | 9 | 2020-09-25 06:00:00.000 | 60.00 | 196,851.90 | 2,738,048.20 | 1,290.92 |
| 63638 | 10 | 38 | 2021-05-15 06:00:00.000 | 28.00 | 167,406.55 | 2,236,218.42 | 1,235.80 |
| 87872 | 5 | 9 | 2023-08-28 06:00:00.000 | 41.00 | 69,283.85 | 769,635.90 | 1,010.84 |
| 36921 | 8 | 14 | 2019-08-16 06:00:00.000 | 28.00 | 58,218.21 | 630,584.03 | 983.14 |

## Extreme Negative Rows

Rows below `-100%` delta are driven by negative actual burn. These should be modeled as corrections/reversals or flagged separately.

| ITEMID | Seq | Num | Date | Interval days | Monthly base | This burn | Delta % |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 75397 | 1 | 13 | 2022-01-15 07:00:00.000 | 30.00 | 39,326.76 | -258,055.54 | -756.18 |
| 57300 | 9 | 9 | 2023-09-12 06:00:00.000 | 992.96 | 196,851.90 | -851,893.50 | -532.76 |
| 21164 | 25 | 25 | 2018-03-14 06:00:00.000 | 99.96 | 163,732.43 | -671,656.35 | -510.22 |
| 77138 | 9 | 9 | 2022-10-29 06:00:00.000 | 63.00 | 267,539.22 | -929,900.00 | -447.58 |
| 58603 | 4 | 12 | 2020-11-09 07:00:00.000 | 9.04 | 57,101.15 | -160,357.17 | -380.83 |
| 36188 | 23 | 23 | 2022-03-07 07:00:00.000 | 984.04 | 27,439.42 | -75,728.38 | -375.98 |
| 17383 | 27 | 30 | 2018-07-21 06:00:00.000 | 1.00 | 69,641.47 | -188,903.65 | -371.25 |
| 63537 | 3 | 8 | 2021-01-20 07:00:00.000 | 69.00 | 51,401.87 | -137,500.00 | -367.50 |
| 41320 | 9 | 10 | 2020-03-12 06:00:00.000 | 174.00 | 77,356.08 | -202,736.02 | -362.08 |
| 36218 | 68 | 68 | 2022-03-07 07:00:00.000 | 993.04 | 11,463.70 | -29,444.69 | -356.85 |
| 19679 | 14 | 27 | 2018-05-19 06:00:00.000 | 28.00 | 121,830.16 | -304,763.40 | -350.15 |
| 17423 | 23 | 24 | 2018-12-27 07:00:00.000 | 8.00 | 26,524.55 | -61,564.24 | -332.10 |
| 44999 | 41 | 66 | 2020-09-29 06:00:00.000 | 1.00 | 24,002.40 | -48,333.60 | -301.37 |
| 99400 | 18 | 18 | 2025-01-16 07:00:00.000 | 95.04 | 284,351.16 | -524,573.57 | -284.48 |
| 92749 | 29 | 45 | 2025-03-14 06:00:00.000 | 1.00 | 420,033.53 | -693,377.24 | -265.08 |

## Most Volatile Items by Row/Monthly Ratio

| ITEMID | Rows | Total burn | Mean actual/monthly | Std dev actual/monthly | Max actual/monthly | Negative rows |
| --- | --- | --- | --- | --- | --- | --- |
| 57300 | 9 | 8,484,317.00 | 4.7889 | 6.2617 | 17.1755 | 1 |
| 80920 | 7 | 2,138,098.03 | 2.4762 | 5.5268 | 16.0091 | 0 |
| 19670 | 12 | 3,150,000.00 | 2.6167 | 5.1305 | 19.1656 | 0 |
| 61751 | 11 | 2,670,225.78 | 3.7394 | 5.0837 | 18.9473 | 2 |
| 68867 | 8 | 660,607.58 | 3.1583 | 5.0817 | 16.2490 | 0 |
| 19672 | 15 | 3,379,129.00 | 2.0933 | 4.7578 | 19.3463 | 2 |
| 44983 | 11 | 1,197,670.03 | 2.7424 | 4.7407 | 17.4446 | 0 |
| 75397 | 13 | 735,410.48 | 1.4385 | 4.6814 | 16.0222 | 1 |
| 71522 | 14 | 6,032,000.00 | 2.9381 | 4.1330 | 14.8608 | 0 |
| 73938 | 13 | 1,138,448.00 | 1.8538 | 3.6766 | 14.4335 | 0 |
| 87855 | 8 | 1,627,896.38 | 3.3417 | 3.2753 | 9.8385 | 1 |
| 87872 | 9 | 1,413,390.63 | 2.2667 | 3.2466 | 11.1084 | 2 |
| 19669 | 14 | 2,220,000.00 | 2.2429 | 3.1832 | 10.7888 | 1 |
| 36921 | 14 | 1,306,028.57 | 1.6024 | 2.8326 | 10.8314 | 1 |
| 55496 | 12 | 1,063,170.00 | 1.5833 | 2.7858 | 9.6504 | 0 |
| 64958 | 15 | 1,118,969.66 | 1.4822 | 2.7629 | 8.5220 | 0 |
| 19671 | 15 | 2,450,000.00 | 2.0933 | 2.7330 | 10.2101 | 1 |
| 64963 | 13 | 778,751.49 | 1.6205 | 2.7191 | 10.5333 | 0 |
| 69763 | 7 | 990,000.00 | 4.0762 | 2.6884 | 7.1333 | 0 |
| 69765 | 7 | 505,000.00 | 4.0762 | 2.6884 | 7.1333 | 0 |

## Recommended Model

A good model for this data should be **hierarchical, scale-aware, and probabilistic**.

Recommended structure:

1. **Item scale layer:** Estimate or accept an item-level total burn and duration. In this retrospective file, `CI_BURN_RATE_MONTHS` supplies that scale. In a forward model, estimate scale from contract item quantity, unit price, item family, project/contract metadata, and schedule information.
2. **Exposure layer:** Convert payment timing into exposure. The simplest exposure is `interval_days / 30`, with first event exposure set to one month. This gives a total-conserving allocation.
3. **Event-shape layer:** Add a multiplicative factor for payment sequence, calendar progress, item duration, and number of paygroups. Use robust medians or a regularized model because the distribution is heavy-tailed.
4. **Correction layer:** Treat negative burns separately with a classifier or flag. They are real behavior but should not be forced into the same continuous positive-burn distribution.
5. **Uncertainty layer:** Predict quantiles or intervals, not just means. The tail is too large for a single point estimate to be honest.

## Practical Formula

For retrospective allocation/anomaly detection:

```text
base_monthly = CI_BURN_RATE_MONTHS
exposure_months = 1.0 for first payment, else days_since_prior_payment / 30
structural_pred = base_monthly * exposure_months
row_pred = structural_pred * sequence_residual_factor
residual = THIS_BURN - row_pred
```

For robust row-level expected payment size when conservation is less important:

```text
row_pred = CI_BURN_RATE_MONTHS * median_ratio(sequence_bucket, item_size_bucket)
```

Use the first formula when item-total reconciliation matters. Use the second when the goal is the typical payment-event amount. For forecasting, replace `CI_BURN_RATE_MONTHS` with a separately estimated item scale to avoid leaking completed-item totals.

## Bottom Line

The burn process is not well described by evenly spreading monthly burn across payment rows. Payment rows are bursty accounting events. The strongest defensible model is a two-level model: estimate item-level scale, then allocate or predict payment-event burn as a noisy, heavy-tailed share of that scale using exposure and sequence features. Negative burns should be explicit correction events, not ordinary low burns.